# Calculus and Optimization Foundations Lab

## Stable functions, derivatives, gradients, Jacobians, chain rules, and descent

This guided lab accompanies the [calculus and optimization prerequisite track](https://github.com/jjames/llm-wiki/tree/main/lessons/prerequisites/02-calculus-optimization).

**Recommended order:** prerequisite lessons → this notebook → Notebook 17 autodiff → Notebook 14 mastery  
**Time:** 90–120 minutes  
**Dependencies:** NumPy and Matplotlib only

### Learning goals

You will connect one-variable derivatives to directional change, compute gradients and Jacobians with shape discipline, verify a multistage chain rule, and diagnose gradient descent from the curvature of a loss surface.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(19)


## 1. Functions, logarithms, and numerical stability

Mathematically equivalent formulas need not be computationally equivalent. Exponentials overflow for large positive inputs; probabilities can underflow; subtracting nearly equal numbers loses precision.

The log-sum-exp identity

$$\log\sum_i e^{z_i}=m+\log\sum_i e^{z_i-m},\quad m=\max_i z_i$$

keeps all exponentials at most one. Stable softmax uses the same shift, which does not change the mathematical probabilities.


In [ ]:
def logsumexp(logits):
    logits = np.asarray(logits, dtype=float)
    maximum = np.max(logits)
    return maximum + np.log(np.exp(logits - maximum).sum())

def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits)
    probabilities = np.exp(shifted)
    return probabilities / probabilities.sum()

logits = np.array([1000.0, 1001.0, 999.0])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(logits) / np.exp(logits).sum()
stable = softmax(logits)

assert not np.all(np.isfinite(naive))
assert np.all(np.isfinite(stable))
np.testing.assert_allclose(stable.sum(), 1.0)
np.testing.assert_allclose(softmax(logits + 5000), stable)
np.testing.assert_allclose(logsumexp(logits), 1001 + np.log(np.exp(-1) + 1 + np.exp(-2)))

print("naive softmax:", naive)
print("stable softmax:", stable)
print("log-sum-exp:", logsumexp(logits))


## 2. Derivatives and directional change

A derivative is a local linear approximation:

$$f(x+h)\approx f(x)+f'(x)h.$$

For a scalar function of a vector, the gradient collects partial derivatives. Its dot product with a unit direction $u$ is the directional derivative $D_uf(x)=\nabla f(x)^Tu$.


In [ ]:
def scalar_function(x):
    return np.sin(x) + 0.1 * x**3

def scalar_derivative(x):
    return np.cos(x) + 0.3 * x**2

x0 = 1.2
steps = np.logspace(-1, -14, 14)
derivative_errors = np.array([
    abs((scalar_function(x0 + h) - scalar_function(x0 - h)) / (2 * h) - scalar_derivative(x0))
    for h in steps
])

def surface(point):
    x, y = point
    return 0.5 * x**2 + 2.0 * y**2 + x * y

point = np.array([1.5, -0.5])
gradient = np.array([point[0] + point[1], 4 * point[1] + point[0]])
direction = np.array([2.0, 1.0]); direction /= np.linalg.norm(direction)
h = 1e-5
numeric_directional = (surface(point + h * direction) - surface(point - h * direction)) / (2 * h)
analytic_directional = gradient @ direction

assert derivative_errors.min() < 1e-8
np.testing.assert_allclose(numeric_directional, analytic_directional, rtol=1e-9, atol=1e-9)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(steps, derivative_errors, marker="o")
ax.set(xlabel="finite-difference step", ylabel="absolute error", title="Derivative checks have a useful step-size range")
ax.grid(True, which="both", alpha=0.25)
plt.show()
print(f"directional derivative: numeric={numeric_directional:.6f}, gradient dot direction={analytic_directional:.6f}")


## 3. A multistage chain rule

Consider

$$z=Ax+b,\quad h=\tanh z,\quad L=\tfrac12\lVert h-y\rVert^2.$$

Work backward from scalar loss to input:

$$\frac{\partial L}{\partial h}=h-y,\quad
\frac{\partial L}{\partial z}=(h-y)\odot(1-h^2),\quad
\nabla_xL=A^T\frac{\partial L}{\partial z}.$$

Every multiplication follows from a Jacobian shape, not from pattern matching.


In [ ]:
A = np.array([[1.0, -2.0], [0.5, 1.5], [-1.0, 0.3]])
b = np.array([0.2, -0.1, 0.4])
target = np.array([0.1, -0.4, 0.7])

def network_loss(x):
    hidden = np.tanh(A @ x + b)
    return 0.5 * np.sum((hidden - target) ** 2)

x = np.array([0.6, -0.2])
z = A @ x + b
hidden = np.tanh(z)
grad_z = (hidden - target) * (1.0 - hidden**2)
analytic_grad_x = A.T @ grad_z

numeric_grad_x = np.zeros_like(x)
epsilon = 1e-5
for index in range(len(x)):
    step = np.zeros_like(x); step[index] = epsilon
    numeric_grad_x[index] = (network_loss(x + step) - network_loss(x - step)) / (2 * epsilon)

assert grad_z.shape == (3,)
assert analytic_grad_x.shape == x.shape
np.testing.assert_allclose(analytic_grad_x, numeric_grad_x, rtol=1e-9, atol=1e-9)
print("analytic gradient:", analytic_grad_x)
print("numeric gradient: ", numeric_grad_x)


## 4. Jacobians: derivatives of vector-valued functions

For $f:\mathbb{R}^n\to\mathbb{R}^m$, the Jacobian has shape $m\times n$. Each row is the gradient of one output. Multiplying $Ju$ predicts how all outputs change along input direction $u$.


In [ ]:
def vector_function(point):
    x, y = point
    return np.array([x * y + x**2, np.exp(x - y), np.sin(y)])

point = np.array([0.8, -0.3])
x, y = point
jacobian = np.array([
    [y + 2 * x, x],
    [np.exp(x - y), -np.exp(x - y)],
    [0.0, np.cos(y)],
])

numeric_jacobian = np.zeros((3, 2))
epsilon = 1e-5
for column in range(2):
    step = np.zeros(2); step[column] = epsilon
    numeric_jacobian[:, column] = (vector_function(point + step) - vector_function(point - step)) / (2 * epsilon)

direction = np.array([0.4, -0.7])
np.testing.assert_allclose(jacobian, numeric_jacobian, rtol=1e-9, atol=1e-9)
np.testing.assert_allclose(jacobian @ direction, numeric_jacobian @ direction, rtol=1e-9, atol=1e-9)
print("Jacobian shape:", jacobian.shape)
print(jacobian)


## 5. Gradient descent and curvature

For the quadratic $L(x)=\tfrac12x^TQx$, the gradient is $Qx$. If $Q$ is positive definite, curvature along its largest-eigenvalue direction limits the stable learning rate:

$$0<\eta<\frac{2}{\lambda_{\max}}.$$

Anisotropic curvature creates zig-zagging: one step size must serve both steep and shallow directions.


In [ ]:
Q = np.array([[1.0, 0.0], [0.0, 20.0]])

def quadratic_loss(x):
    return 0.5 * x @ Q @ x

def run_gradient_descent(learning_rate, steps=40):
    x = np.array([4.0, 1.5])
    trajectory = [x.copy()]
    losses = [quadratic_loss(x)]
    for _ in range(steps):
        x = x - learning_rate * (Q @ x)
        trajectory.append(x.copy()); losses.append(quadratic_loss(x))
    return np.array(trajectory), np.array(losses)

stable_trajectory, stable_losses = run_gradient_descent(0.06)
unstable_trajectory, unstable_losses = run_gradient_descent(0.11)
assert stable_losses[-1] < 1e-2 * stable_losses[0]
assert unstable_losses[-1] > unstable_losses[0]
assert 0.06 < 2 / np.linalg.eigvalsh(Q).max() < 0.11

grid_x = np.linspace(-4.5, 4.5, 200)
grid_y = np.linspace(-1.8, 1.8, 200)
X, Y = np.meshgrid(grid_x, grid_y)
Z = 0.5 * (X**2 + 20 * Y**2)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].contour(X, Y, Z, levels=np.geomspace(0.05, 40, 12))
axes[0].plot(stable_trajectory[:, 0], stable_trajectory[:, 1], marker="o", markersize=3, label="η=0.06")
axes[0].set(title="Stable but zig-zagging", xlabel="x₁", ylabel="x₂"); axes[0].legend()
axes[1].semilogy(stable_losses, label="η=0.06")
axes[1].semilogy(unstable_losses, label="η=0.11")
axes[1].set(title="The curvature limit is sharp", xlabel="step", ylabel="loss"); axes[1].legend(); axes[1].grid(alpha=0.25)
plt.show()
print("eigenvalues of Q:", np.linalg.eigvalsh(Q), "stable η must be below", 2 / np.linalg.eigvalsh(Q).max())


## Cumulative investigations

1. Add a constant to every logit. Prove why softmax is unchanged.
2. Draw the gradient at `point` and identify the direction of steepest decrease.
3. Re-derive `analytic_grad_x` using explicit Jacobian matrices and check every shape.
4. Explain why a Jacobian is not “the gradient” unless the output is scalar.
5. Rotate the quadratic without changing its eigenvalues. Does the stability bound change?
6. Try learning rates `0.099` and `0.101`; explain the qualitative difference.

**Next:** Notebook 17 builds forward and reverse automatic differentiation from scratch. Notebook 14 then asks you to implement and compare optimizers with fewer hints.
